# Vianello Gut Transcriptomic Analysis & Export Pipeline
**Project:** Vianello et al. 2026 Replication & Sub-Categorical Filtering Workspace
**Authors:** Alina Krawczynski, Rannyele Passos Ribeiro (powered by Claude)
**Lab:** Özpolat Lab, Washington University in St. Louis, 1 Brookings Drive, St. Louis, MO 63130
**Year:** 2026

---

## Dataset Overview & Biological Context
This application is anchored strictly within **`VST_Pd_corrected.rds`**, the variance-stabilized, batch-corrected expression matrix for *Platynereis dumerilii* (Pd) gut segments from the Vianello et al. 2026 comparative gut transcriptomics study (`doi:10.5281/zenodo.17746910`). This is the master matrix used for all downstream plotting.

### Understanding the Expression Values
* **Generation:** Raw reads were mapped to the *Platynereis dumerilii* reference and counted per transcript (`counts_Pd.rds`, 69,611 features x 30 samples).
* **Normalization:** Counts were Variance-Stabilizing-Transformed (VST) and batch-corrected across the 30 samples (10 gut segments x 3 replicates), yielding `37,392` reliable transcripts. In plain terms, VST re-scales each count so its variance no longer depends on its average expression level (raw RNA-seq counts are naturally noisier for highly-expressed genes) — this keeps low- and high-abundance transcripts equally comparable and is a standard preprocessing step before clustering or visualization.
* **Interpretation:** Values represent **relative transcript abundance** along the gut's anterior-to-posterior axis, enabling direct comparison across segments.

### Automated Data Cleaning & QC Rules Applied:
1. **Index Standardization:** All transcript identifiers are coerced to strings for consistent lookups.
2. **Replicate Averaging:** Each gut segment's 3 biological replicates (columns like `Pd_mouth_1/2/3`) are collapsed into a single mean value per segment, producing 10 clean condition columns per gene.
3. **Anatomical Ordering:** Segments are always displayed in their true anterior to posterior order (`mouth -> stomodeum -> transition1 -> esophagus -> transition2 -> stomach -> transition3 -> midgut -> transition4 -> hindgut`), taken directly from the source matrix's own column ordering.

---

## Key Features
* **Hybrid Gene Input:** Search transcripts dynamically using a live-filtering list (hold `Ctrl`/`Cmd` to multi-select), or paste a raw block of `XLOC_...` IDs separated by commas or spaces.
* **Flexible Segment Filtering:** Isolate a single gut segment, or hold `Ctrl`/`Cmd` to select any custom combination of segments.
* **Dual-Source Annotation:** Each searched gene is checked against both the curated **Transcription Factor registry** (`Pd_TF_ID.xlsx`) and the broader **functional dictionary** (`Pd_dictionary_standard.rds`), matching on the shared `XLOC_...` gene ID.
* **On-Demand Exporter:** A dual-button layout separates quick visualization from storage. Click the blue button to write vector-based `.pdf` plots and clean `.csv` annotations to your chosen folder.
* **Multi-Gene Heatmap:** Selecting 2+ genes automatically adds a genes-by-segment, hierarchically clustered heatmap (row-normalized so genes at different expression scales stay visually comparable) with a colorblind-safe palette toggle, after the individual bar charts, included in the PDF export.

---

## Dashboard User Guide

### Step 1: Choose Gut Segments
Hold `Ctrl` (or `Cmd` on Mac) in the **Segments** box to highlight any custom combination of the 10 gut regions. All segments are selected by default.

### Step 2: Choose Target Transcripts (Use A, B, or C!)
* **Option A (Filter List):** Type into **Filter List** to shrink the selection panel in real-time, then click (or Ctrl/Cmd-click) transcript IDs.
* **Option B (Paste Box):** Drop a raw text block of `XLOC_...` IDs copied from a spreadsheet straight into the box.
* **Option C (Search by Keyword):** type a keyword to find every gene whose TF-registry or dictionary annotation contains it.

### Step 3: Run and/or Save
* Click **🎯 Generate Visuals (View Only)** to render bar charts, the multi-gene heatmap, and annotation tables instantly on-screen.
* Click **💾 Export Active Results (Save CSV/PDF)** to save high-resolution vector assets to your chosen folder (defaults to `exported_results/`).


In [ ]:
# ============================================================================================
#  DEPENDENCY CHECK -- run this cell first.
#  Installs any Python packages this notebook needs that aren't already present, then you're
#  done. Safe to re-run (does nothing if everything is already installed). To install by hand
#  instead, run this in a terminal / Anaconda Prompt:
#      pip install pandas numpy matplotlib seaborn scipy ipywidgets openpyxl pyreadr
# ============================================================================================
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'ipywidgets', 'openpyxl', 'pyreadr']

_missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("\nInstall finished. If an import fails in a later cell, restart the kernel")
    print("(menu: Kernel -> Restart Kernel) and run the notebook again from the top.")
else:
    print("All required packages are already installed -- you're good to go.")

In [ ]:
import os
import pandas as pd
import numpy as np
import pyreadr
import matplotlib.pyplot as plt
import seaborn as sns

def load_rds(path):
    result = pyreadr.read_r(path)
    return result[list(result.keys())[0]]

# 1. Load Master Expression Matrix (VST-normalized, batch-corrected)
df_vst = load_rds("VST_Pd_corrected.rds")
df_vst.index = df_vst.index.astype(str)

# 2. Load Sample Metadata (maps Pd_<segment>_<rep> columns -> segment/replicate)
df_meta = load_rds("Pd_samplemetadata_standard.rds")
df_meta.index = df_meta.index.astype(str)

# 3. Load Annotation Sources
df_tf = pd.read_excel("Pd_TF_ID.xlsx")
df_tf["ID"] = df_tf["ID"].astype(str)

df_dict = load_rds("Pd_dictionary_standard.rds")
df_dict["geneID"] = df_dict["geneID"].astype(str)
# Keep first annotation row per gene so lookups stay 1:1
df_dict_unique = df_dict.drop_duplicates(subset="geneID", keep="first").set_index("geneID")

# 4. Anatomical (anterior -> posterior) segment order, taken from the matrix's own column ordering
SEGMENT_ORDER = ["mouth", "stomodeum", "transition1", "esophagus", "transition2",
                  "stomach", "transition3", "midgut", "transition4", "hindgut"]

# 5. Collapse the 3 replicates into a single mean value per gut segment
df_mean = pd.DataFrame(index=df_vst.index)
for seg in SEGMENT_ORDER:
    seg_cols = [c for c in df_vst.columns if c.startswith(f"Pd_{seg}_")]
    df_mean[seg] = df_vst[seg_cols].mean(axis=1)

print(f"✅ Master Matrix Ready. {len(df_mean)} transcripts across {len(SEGMENT_ORDER)} gut segments (replicate-averaged).")


---

## 🔬 BLAST Sequence Search
Have a nucleotide sequence from elsewhere and want to know if/where it shows up in this dataset? Paste it
below to BLAST it against the **pdumv021 reference transcriptome** (provided directly by the PI — a local
`blastn` search, no internet required). This is the exact transcript-model assembly that produced this
dataset's own `XLOC_...`/`TCONS_...` gene IDs, so a match here maps directly onto a searchable gene in the
dashboard below. Any matching gene ID found can be copied into the **Paste Box** in Step 2 to view its
expression profile.

**Setup (optional — the rest of the dashboard works without this):** the sequence search needs
the free NCBI BLAST+ toolkit plus a small search database you build once from the reference
transcriptome. Step-by-step instructions — install BLAST+, get the reference FASTA, and build
the database into a `blast_db/` folder next to this notebook — are in the comment at the top of
the next code cell. The notebook finds BLAST+ automatically once it is installed; until then this
section just prints a short reminder and everything else in the notebook is unaffected.

In [ ]:
import shutil
import subprocess
import tempfile

# --- Sequence -> Gene ID search: paste any raw nucleotide sequence and find the best-matching
# transcript(s) in the pdumv021 reference transcriptome via a local blastn search. This is the
# exact assembly that produced this dashboard's own XLOC_.../TCONS_... gene IDs (confirmed: headers
# already carry "TCONS_... XLOC_..." directly), so hits map straight onto searchable genes above.
#
# SETUP (only needed once per machine): install BLAST+ from
# https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/ , then build the database directly from
# pdumv021.transcripts.fasta (no header rewriting needed -- TCONS_... is already the first token):
#   makeblastdb -in pdumv021.transcripts.fasta -dbtype nucl -parse_seqids -out blast_db/vianello_transcriptome
# The notebook auto-detects BLAST+ on your PATH (or set BLAST_BIN_DIR
# below) and looks for the database in ./blast_db/. Once
# built, the database is self-contained -- the original FASTA isn't needed for searching or export.

# --- BLAST+ location (auto-detected) ---------------------------------------------------------
# By default this notebook finds the BLAST+ programs (blastn, blastdbcmd, makeblastdb) on your
# system PATH -- if you installed BLAST+ the normal way, leave BLAST_BIN_DIR = "" and it just
# works. If BLAST+ lives somewhere not on your PATH, set BLAST_BIN_DIR to the folder that holds
# the blastn / blastdbcmd / makeblastdb executables.
BLAST_BIN_DIR = ""   # e.g. r"C:\Program Files\NCBI\blast-2.17.0+\bin"  or  "/usr/local/ncbi/blast/bin"

# The prebuilt search database: a set of files sharing one name prefix (<name>.nin, .nhr, .nsq,
# ...). Defaults to a "blast_db" folder next to this notebook. Build it once from the reference
# transcriptome -- see the SETUP note above. This whole BLAST section is optional; the rest of
# the dashboard works fine without it.
BLAST_DB_PATH = os.path.join("blast_db", "vianello_transcriptome")

def _find_blast_tool(tool_name):
    """Locate a BLAST+ executable: honour BLAST_BIN_DIR if it is set, otherwise search the
    system PATH. Returns the full path or None. shutil.which adds the right extension (.exe)
    on Windows automatically."""
    if BLAST_BIN_DIR:
        found = shutil.which(tool_name, path=BLAST_BIN_DIR)
        if found:
            return found
    return shutil.which(tool_name)

# Cache of the most recent search, so the export button can act on whatever was last searched
_CURRENT_BLAST_QUERY = None
_CURRENT_BLAST_HITS = None

def _to_fasta(raw_text):
    """Wrap raw pasted text into a single-record FASTA query -- accepts either bare sequence
    (any whitespace/line breaks) or an already-FASTA-formatted block with its own header."""
    raw_text = raw_text.strip()
    if raw_text.startswith(">"):
        return raw_text
    seq = "".join(raw_text.split())
    return f">user_query\n{seq}"

def run_blast_search(raw_sequence, max_hits=10):
    """BLASTs a pasted nucleotide sequence against the pdumv021 transcriptome and returns a
    results DataFrame with one row per hit. Gene_ID is read straight from the header's second
    token (the XLOC_... locus ID) -- no regex parsing needed since the header already separates
    transcript ID (first token) from gene ID (second token) cleanly."""
    blastn_exe = _find_blast_tool("blastn")
    if blastn_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    if not os.path.isfile(BLAST_DB_PATH + ".nin"):
        print(f"❌ BLAST database not found at {BLAST_DB_PATH} -- see the setup note above this cell.")
        return None

    fasta_query = _to_fasta(raw_sequence)
    seq_only = "".join(line for line in fasta_query.splitlines() if not line.startswith(">"))
    if len(seq_only) < 15:
        print("❌ Query sequence looks too short (or empty) to BLAST -- paste at least ~15 bases.")
        return None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
        f.write(fasta_query)
        query_path = f.name

    try:
        result = subprocess.run(
            [blastn_exe, "-query", query_path, "-db", BLAST_DB_PATH,
             "-outfmt", "6 sseqid pident length evalue bitscore stitle",
             "-max_target_seqs", str(max_hits)],
            capture_output=True, text=True, timeout=60
        )
    finally:
        os.remove(query_path)

    if result.returncode != 0:
        print("❌ blastn reported an error:")
        print(result.stderr.strip())
        return None

    lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
    if not lines:
        print("🛑 No BLAST hits found for this sequence in the pdumv021 transcriptome.")
        return None

    rows = []
    for line in lines:
        sseqid, pident, length, evalue, bitscore, stitle = line.split("\t", 5)
        gene_id = stitle.strip()
        rows.append({
            "Gene_ID": gene_id,
            "In_Dashboard": gene_id in df_mean.index,
            "Transcript_Hit": sseqid,
            "%Identity": float(pident),
            "Alignment_Length": int(length),
            "E-value": float(evalue),
            "Bit_Score": float(bitscore),
        })

    df_hits = pd.DataFrame(rows).sort_values("Bit_Score", ascending=False).reset_index(drop=True)
    return df_hits

def get_sequence_from_db(seq_id):
    """Retrieves a sequence's full FASTA record directly from the BLAST database via blastdbcmd
    -- works entirely from the compiled database, without needing the original FASTA on disk."""
    blastdbcmd_exe = _find_blast_tool("blastdbcmd")
    if blastdbcmd_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    result = subprocess.run(
        [blastdbcmd_exe, "-db", BLAST_DB_PATH, "-entry", seq_id],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

def export_blast_hits(query_sequence, df_hits, save_dir="exported_results"):
    """Saves the current BLAST hit table (CSV) and the matched sequences, retrieved straight
    from the database, as a multi-FASTA file (also including the original query sequence for
    reference) to save_dir."""
    if df_hits is None or df_hits.empty:
        print("❌ Nothing to export -- run a BLAST search with results first!")
        return

    os.makedirs(save_dir, exist_ok=True)
    df_hits.to_csv(os.path.join(save_dir, "blast_hits.csv"), index=False)
    print(f"💾 [SAVED] BLAST hits table exported to '{save_dir}/blast_hits.csv'")

    fasta_records = [_to_fasta(query_sequence).replace(">user_query", ">query_sequence", 1)]
    for seq_id in df_hits["Transcript_Hit"]:
        record = get_sequence_from_db(seq_id)
        if record:
            fasta_records.append(record)

    with open(os.path.join(save_dir, "blast_hit_sequences.fasta"), "w") as f:
        f.write("\n".join(fasta_records) + "\n")
    print(f"💾 [SAVED] Query + {len(fasta_records) - 1} hit sequence(s) exported to '{save_dir}/blast_hit_sequences.fasta'")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker (tkinter, ships with Python) so the user can point at an
    existing folder directly instead of typing/remembering a path -- falls back to leaving the
    text box untouched if the dialog is cancelled."""
    def _browse(b):
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        folder = filedialog.askdirectory(title="Choose a folder to save results into")
        root.destroy()
        if folder:
            target_box.value = folder
    return _browse

blast_query_box = widgets.Textarea(
    placeholder="Paste a raw nucleotide sequence here (with or without a '>' FASTA header)...",
    description="Sequence:",
    layout=widgets.Layout(width="600px", height="120px")
)
blast_button = widgets.Button(description="🔬 Run BLAST Search", button_style="warning", layout=widgets.Layout(width="270px", height="40px"))
blast_export_button = widgets.Button(description="💾 Export Hit Sequences", button_style="info", layout=widgets.Layout(width="270px", height="40px"))
blast_save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
blast_browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
blast_browse_button.on_click(_make_browse_handler(blast_save_location_box))
blast_output = widgets.Output()

def on_blast_clicked(b):
    global _CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS
    with blast_output:
        clear_output(wait=True)
        raw_sequence = blast_query_box.value.strip()
        if not raw_sequence:
            print("❌ Paste a sequence first!")
            return

        print("🔬 Running BLAST search against the pdumv021 transcriptome...")
        df_hits = run_blast_search(raw_sequence)
        _CURRENT_BLAST_QUERY = raw_sequence
        _CURRENT_BLAST_HITS = df_hits
        if df_hits is None:
            return

        print(f"\n✅ {len(df_hits)} hit(s) found (best match first):")
        display(df_hits)

        best_ids = df_hits.loc[df_hits["In_Dashboard"], "Gene_ID"].dropna().unique().tolist()
        if best_ids:
            print(f"\n📋 Copy into the 'Paste Box' below to view expression profiles: {', '.join(best_ids)}")
        else:
            print("\n⚠️ None of the matched transcripts have a corresponding entry in the expression matrix below.")

def on_blast_export_clicked(b):
    with blast_output:
        clear_output(wait=True)
        save_dir = blast_save_location_box.value.strip() or "exported_results"
        export_blast_hits(_CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS, save_dir=save_dir)

blast_button.on_click(on_blast_clicked)
blast_export_button.on_click(on_blast_export_clicked)

print("🔬 BLAST Sequence Search Ready:")
display(widgets.VBox([
    widgets.HTML("<i>Paste a sequence of interest below to find its matching gene ID in this dataset.</i>"),
    blast_query_box,
    widgets.HBox([blast_save_location_box, blast_browse_button]),
    widgets.HBox([blast_button, blast_export_button]),
    blast_output
]))

In [ ]:
import re
import numpy as np
from IPython.display import display, HTML

# Global tracking variables so the export button knows what is currently active
_CURRENT_GENES = []
_CURRENT_SEGMENTS = []

def _safe_filename(gene_id):
    """Strips characters Windows forbids in filenames (some gene IDs, e.g. mitochondrial ones
    like 'NC_000931.1:2353..2415', contain a ':' that otherwise crashes plt.savefig)."""
    return re.sub(r'[\\/:*?"<>|]', "_", gene_id)

def _bar_palette(cmap, n):
    """Builds the palette used for the per-condition bars. The default keeps the original 'mako'
    look (independent of the heatmap's red-blue default) since bar color here is purely
    decorative, not tied to a z-score axis. Non-default (colorblind) choices reuse the same
    diverging colormap picked for the heatmap, sampled from its saturated half only -- diverging
    colormaps (PuOr, BrBG) have a washed-out near-white color at their exact center, which would
    otherwise make a bar look broken/invisible if landed on."""
    if cmap == "RdBu_r":
        return "mako"
    cmap_obj = plt.get_cmap(cmap)
    if cmap in ("PuOr", "BrBG"):
        positions = np.linspace(0.60, 0.95, max(n, 1))
    else:
        positions = np.linspace(0.15, 0.85, max(n, 1))
    return [cmap_obj(p) for p in positions]

def _lookup_annotation(gene_id):
    """Check the TF registry first, then fall back to the broader functional dictionary."""
    rows = []
    tf_hit = df_tf[df_tf["ID"] == gene_id]
    for _, r in tf_hit.iterrows():
        rows.append({"Gene_ID": gene_id, "Source": "TF Registry", "Symbol_or_Family": r["TFs"], "Description": ""})

    if gene_id in df_dict_unique.index:
        d = df_dict_unique.loc[gene_id]
        symbol = d["symbol"] if isinstance(d["symbol"], str) else ""
        desc = d["description"] if isinstance(d["description"], str) else ""
        # Skip placeholder rows where symbol/description just repeat the raw gene ID (i.e. unannotated)
        if symbol != gene_id or desc != gene_id:
            rows.append({"Gene_ID": gene_id, "Source": "Pd Dictionary", "Symbol_or_Family": symbol, "Description": desc})

    return rows

def plot_multi_genes(gene_list, segments_chosen=None):
    """Resolves the gene/segment selection and stores it in _CURRENT_GENES/_CURRENT_SEGMENTS.
    Call _render_bar_charts()/_print_annotations() afterward to actually display output --
    split out so the caller controls display order (heatmap, then plots, then annotations)."""
    global _CURRENT_GENES, _CURRENT_SEGMENTS

    segments = [s for s in SEGMENT_ORDER if not segments_chosen or s in segments_chosen]
    if not segments:
        segments = SEGMENT_ORDER

    _CURRENT_GENES = [g.strip() for g in gene_list if g.strip() in df_mean.index]
    _CURRENT_SEGMENTS = segments

def _render_bar_charts(cmap="RdBu_r"):
    for gene_id in _CURRENT_GENES:
        gene_data = df_mean.loc[gene_id][_CURRENT_SEGMENTS]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(
            x=gene_data.index,
            y=gene_data.values.astype(float),
            hue=gene_data.index,
            palette=_bar_palette(cmap, len(gene_data.index)),
            legend=False,
            width=0.6
        )

        plt.title(f"Expression Profile: {gene_id}", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("VST-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Gut Segment (Anterior → Posterior)", fontsize=14, fontweight="bold")
        plt.xticks(rotation=30, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()
        plt.show()

def _print_annotations(gene_list):
    print("\n📋 GENE ANNOTATION RESULTS:")
    for gene_id in gene_list:
        gene_id = gene_id.strip()
        rows = _lookup_annotation(gene_id)
        if rows:
            print(f"\nAnnotations for Search ID: {gene_id}")
            display(pd.DataFrame(rows))
        else:
            print(f"\nAnnotations for Search ID: {gene_id} -> 🛑 [No annotation found]")

def _keyword_regex(keyword):
    """Left-word-boundary pattern: the keyword must start at a word boundary (so 'vasa' won't
    match mid-word inside 'extravasation'), but is still free to be a PREFIX of a longer word
    (so 'kinase' still matches 'kinases', 'kinase-like', 'tyrosine-kinase-1', etc.). A plain
    case-insensitive substring search (what this used to be) has no such boundary check, so a
    short keyword can silently match as a fragment of an unrelated word -- confirmed with a real
    example: searching "Vasa" matched a gene whose only hit was the word "extravasation" in its
    GO description, nothing to do with the Vasa gene."""
    return r"\b" + re.escape(keyword)

def _search_annotations_by_keyword(keyword):
    """Finds every gene whose TF-family label or dictionary symbol/description contains the
    given keyword as a word-initial match (see _keyword_regex), returning gene IDs actually
    searchable in this dashboard (i.e. present in df_mean.index)."""
    keyword = keyword.strip()
    if not keyword:
        return []
    pattern = _keyword_regex(keyword)
    matched = set()

    tf_mask = df_tf["TFs"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched.update(df_tf.loc[tf_mask, "ID"].astype(str))

    dict_mask = (
        df_dict_unique["symbol"].astype(str).str.contains(pattern, case=False, na=False, regex=True) |
        df_dict_unique["description"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    )
    matched.update(df_dict_unique.index[dict_mask].astype(str))

    matched_ids = [g for g in matched if g in df_mean.index]
    return sorted(matched_ids)

# --- SEPARATE EXPORT EXECUTION FUNCTION ---
def export_cached_results(save_dir="exported_results", cmap="RdBu_r"):
    if not _CURRENT_GENES:
        print("❌ Nothing available to save. Please run a successful query search first!")
        return

    os.makedirs(save_dir, exist_ok=True)

    # 1. Export Current Annotations to CSV
    all_rows = []
    for gene_id in _CURRENT_GENES:
        all_rows.extend(_lookup_annotation(gene_id))

    if all_rows:
        pd.DataFrame(all_rows).to_csv(os.path.join(save_dir, "selected_gene_annotations.csv"), index=False)
        print(f"💾 [SAVED] Annotation CSV created inside '{save_dir}/' folder.")

    # 2. Export Current Graphs to Vector PDFs
    for gene_id in _CURRENT_GENES:
        gene_data = df_mean.loc[gene_id][_CURRENT_SEGMENTS]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(x=gene_data.index, y=gene_data.values.astype(float), hue=gene_data.index,
                    palette=_bar_palette(cmap, len(gene_data.index)), legend=False, width=0.6)
        plt.title(f"Expression Profile: {gene_id}", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("VST-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Gut Segment (Anterior → Posterior)", fontsize=14, fontweight="bold")
        plt.xticks(rotation=30, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()

        plt.savefig(os.path.join(save_dir, f"expression_profile_{_safe_filename(gene_id)}.pdf"), format="pdf", dpi=300, bbox_inches="tight")
        plt.close()

    print(f"💾 [SAVED] Vector PDF plots for {_CURRENT_GENES} generated inside '{save_dir}/' folder.")

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage

# --- Multi-gene comparison view: hierarchically-clustered heatmap (Aim E instructions: "generate an
# interactive heatmap..."). Only meaningful with 2+ genes selected, so it's skipped automatically for
# single-gene searches. Both genes (rows) and conditions (columns) are hierarchically clustered so
# genes/conditions with similar profiles end up next to each other.

CMAP_OPTIONS = {
    "Default": "RdBu_r",
    "Purple-Orange (colorblind-safe)": "PuOr",
    "Brown-Teal (colorblind-safe)": "BrBG",
    "Cividis (colorblind-safe, single-hue)": "cividis",
}

def _zscore_matrix(gene_ids, cols, matrix):
    sub = matrix.loc[gene_ids, cols].astype(float)
    # Row z-score so genes with very different absolute expression levels are still
    # visually comparable on the same color scale.
    # NOTE: std() over a single selected column is undefined (NaN), which silently makes the
    # whole z-scored row NaN even though the raw data itself has real values -- this is why
    # validity has to be checked on z (the thing actually plotted), not on the raw matrix.
    row_std = sub.std(axis=1).replace(0, 1)
    z = sub.sub(sub.mean(axis=1), axis=0).div(row_std, axis=0)
    return sub, z

def _render_heatmap(gene_ids, cols, matrix, value_label="Expression", cmap="RdBu_r"):
    sub, z = _zscore_matrix(gene_ids, cols, matrix)

    # Drop any gene/condition with zero valid data among the current selection -- otherwise a
    # fully-blank row or column (common when only some picked genes have data for some of the
    # picked conditions) pads the heatmap out with wasted, confusing empty space.
    z = z.loc[z.notna().any(axis=1), z.notna().any(axis=0)]
    gene_ids = list(z.index)
    cols = list(z.columns)

    # Hierarchical clustering (genes and conditions): reorders rows/columns so similar profiles sit
    # next to each other, shown as dendrograms along the top/left. NaN cells are filled with 0 only
    # for the clustering-distance calculation (scipy's linkage can't handle NaN); the actual displayed
    # cell still renders blank via mask=, so a gene's real missing data is never disguised as a zero.
    z_for_clustering = z.fillna(0)
    row_linkage = linkage(z_for_clustering, method="average") if len(gene_ids) > 1 else None
    col_linkage = linkage(z_for_clustering.T, method="average") if len(cols) > 1 else None

    g = sns.clustermap(
        z, cmap=cmap, center=0, mask=z.isna(),
        row_linkage=row_linkage, col_linkage=col_linkage,
        row_cluster=len(gene_ids) > 1, col_cluster=len(cols) > 1,
        linewidths=0.5, annot=False,
        figsize=(max(6, len(cols) * 0.9) + 2.2, max(4, len(gene_ids) * 1.3 + 2)),
        cbar_pos=(0.90, 0.86, 0.025, 0.10), dendrogram_ratio=(0.15, 0.15),
    )

    # sns.clustermap lets the heatmap/col-dendrogram extend almost to the figure's right edge
    # regardless of cbar_pos (the colorbar floats independently, on top), so it silently overlaps
    # the heatmap's top-right corner. Directly shrink the heatmap + column-dendrogram axes to end
    # before the colorbar's left edge -- NOT via fig.subplots_adjust(), which was found (by direct
    # testing) to trigger an internal seaborn recompute that snaps cbar_pos back to its own default
    # top-left position instead of respecting the value passed in. The colorbar is also placed
    # above the heatmap (not beside its top rows) so it never collides with row labels either.
    cbar_left, margin = 0.90, 0.02
    for ax in (g.ax_heatmap, g.ax_col_dendrogram):
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, (cbar_left - margin) - pos.x0, pos.height])

    g.ax_heatmap.set_xlabel("Gut Segment (Anterior → Posterior)", fontsize=16, fontweight="bold")
    g.ax_heatmap.set_ylabel("")
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right", fontsize=14)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=-30, ha="left", va="center", fontsize=13)
    g.ax_cbar.set_title("Row\nz-score", fontsize=11, fontweight="bold")
    g.ax_cbar.tick_params(labelsize=8)
    g.fig.suptitle(f"Expression Heatmap ({value_label}, hierarchically clustered)",
                   fontsize=17, fontweight="bold", x=0.42, ha="center", y=1.08)
    return g.fig

def _has_any_valid_data(gene_ids, cols, matrix):
    cols = list(dict.fromkeys(cols))  # defensive: guard against any duplicate column labels
    _, z = _zscore_matrix(gene_ids, cols, matrix)
    return bool(z.notna().to_numpy().sum())

def render_multi_gene_views(gene_ids, cols, matrix, value_label="VST-Normalized Expression", cmap="RdBu_r"):
    """Call after plot_multi_genes(): shows the clustered heatmap when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions -- heatmap skipped.")
        return
    print("\n📊 MULTI-GENE COMPARISON HEATMAP:")
    _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    plt.show()

def save_multi_gene_views(gene_ids, cols, matrix, value_label="VST-Normalized Expression", cmap="RdBu_r", save_dir="exported_results"):
    """Call after export_cached_results(): saves the heatmap as a PDF when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions -- heatmap skipped.")
        return
    os.makedirs(save_dir, exist_ok=True)
    fig = _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    fig.savefig(os.path.join(save_dir, "multi_gene_heatmap.pdf"), format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 [SAVED] Heatmap PDF generated inside '{save_dir}/' folder.")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker (tkinter, ships with Python) so the user can point at an
    existing folder directly instead of typing/remembering a path. Defined locally (not reused
    from the BLAST cell above) so this cell stays self-sufficient regardless of cell order."""
    def _browse(b):
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        folder = filedialog.askdirectory(title="Choose a folder to save results into")
        root.destroy()
        if folder:
            target_box.value = folder
    return _browse

# Glossary Accordion View
key_details = widgets.HTML(value='''
<div style="background-color: #f7f9fa; padding: 12px; border-left: 4px solid #2b5b84; border-radius: 4px; font-family: sans-serif; font-size: 13px; line-height: 1.5;">
    <b>📋 GUT SEGMENT DICTIONARY (Anterior → Posterior):</b><br>
    mouth → stomodeum → transition1 → esophagus → transition2 → stomach → transition3 → midgut → transition4 → hindgut<br><br>
    Each bar shown is the <b>mean of 3 biological replicates</b> for that segment (VST-normalized, batch-corrected expression).
</div>
''')
glossary_dropdown = widgets.Accordion(children=[key_details], selected_index=None)
glossary_dropdown.set_title(0, "🔍 Click to view/hide Gut Segment Glossary")
glossary_dropdown.layout.width = "670px"

# Inputs
segment_multiselect = widgets.SelectMultiple(
    options=SEGMENT_ORDER,
    value=tuple(SEGMENT_ORDER),
    description="Segments:",
    layout=widgets.Layout(width="200px", height="160px")
)
cmap_dropdown = widgets.Dropdown(options=list(CMAP_OPTIONS.keys()), description="Palette:", layout=widgets.Layout(width="420px"))

# Filter List: precompute the sorted gene-ID list and a lowercase mirror ONCE here, rather than
# re-sorting + re-lowering all ~37K transcript IDs from df_mean.index inside the keystroke handler
# on every character typed (that used to make typing in this box visibly laggy). The widget is
# also capped to MAX_FILTER_RESULTS entries so a broad search term never asks the browser to
# render tens of thousands of <option> rows at once.
_ALL_GENE_IDS = sorted(df_mean.index.astype(str))
_ALL_GENE_IDS_LOWER = [g.lower() for g in _ALL_GENE_IDS]
MAX_FILTER_RESULTS = 500

search_filter = widgets.Text(placeholder="Type to filter transcripts (e.g. XLOC_0002...)...", description="Filter List:", layout=widgets.Layout(width="440px"))
gene_multiselect = widgets.SelectMultiple(options=_ALL_GENE_IDS[:MAX_FILTER_RESULTS], description="Select Genes:", layout=widgets.Layout(width="440px", height="140px"))
filter_hint = widgets.HTML(value=f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(_ALL_GENE_IDS)} -- type to narrow down.</i>")

def update_multiselect_options(change):
    search_term = change["new"].strip().lower()
    if search_term:
        matches = [gid for gid, gid_lower in zip(_ALL_GENE_IDS, _ALL_GENE_IDS_LOWER) if search_term in gid_lower]
    else:
        matches = _ALL_GENE_IDS
    gene_multiselect.options = matches[:MAX_FILTER_RESULTS]
    if len(matches) > MAX_FILTER_RESULTS:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(matches)} matches -- refine your search to narrow further.</i>"
    else:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>{len(matches)} matching entr{'y' if len(matches) == 1 else 'ies'}.</i>"
search_filter.observe(update_multiselect_options, names="value")

multi_gene_input = widgets.Textarea(placeholder="OR paste multiple Gene IDs here (e.g. XLOC_000013, XLOC_018638)...", description="Paste Box:", layout=widgets.Layout(width="440px", height="100px"))

# --- Option C: Search by Annotation Keyword -- finds every gene whose TF/dictionary annotation
# contains a keyword, then runs it through the exact same Heatmap -> Plots -> Annotation pipeline
# as the other two search options, sharing the same output_area below. ---
keyword_search_box = widgets.Text(placeholder="e.g. homeobox, kinase, Wnt...", description="Keyword:", layout=widgets.Layout(width="250px"))
keyword_search_button = widgets.Button(description="🔎 Search Annotations", button_style="warning", layout=widgets.Layout(width="250px", height="30px"))

MAX_KEYWORD_MATCHES = 30

# TWO SEPARATE BUTTON CONTROLS
submit_button = widgets.Button(description="🎯 Generate Visuals (View Only)", button_style="success", layout=widgets.Layout(width="330px", height="40px"))
export_button = widgets.Button(description="💾 Export Active Results (Save CSV/PDF)", button_style="info", layout=widgets.Layout(width="330px", height="40px"))
save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
browse_button.on_click(_make_browse_handler(save_location_box))

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        chosen_genes = list(gene_multiselect.value)
        pasted_genes = [g.strip() for g in multi_gene_input.value.replace(",", " ").split() if g.strip()]
        final_gene_list = sorted(list(set(chosen_genes + pasted_genes)))

        if not final_gene_list:
            print("❌ Input target genes first!")
            return

        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(final_gene_list, segments_chosen=list(segment_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_SEGMENTS, df_mean, "VST-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(final_gene_list)

def on_export_clicked(b):
    with output_area:
        save_dir = save_location_box.value.strip() or "exported_results"
        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        export_cached_results(save_dir=save_dir, cmap=cmap)
        save_multi_gene_views(_CURRENT_GENES, _CURRENT_SEGMENTS, df_mean, "VST-Normalized Expression", cmap, save_dir=save_dir)

def on_keyword_search_clicked(b):
    with output_area:
        clear_output(wait=True)
        keyword = keyword_search_box.value.strip()
        if not keyword:
            print("❌ Type a keyword first!")
            return

        matched_ids = _search_annotations_by_keyword(keyword)
        if not matched_ids:
            print(f"🛑 No genes found with '{keyword}' in their annotation.")
            return

        print(f"✅ Found {len(matched_ids)} gene(s) matching '{keyword}'.")
        if len(matched_ids) > MAX_KEYWORD_MATCHES:
            print(f"⚠️ Limiting to the first {MAX_KEYWORD_MATCHES} matches to keep the plots manageable.")
            matched_ids = matched_ids[:MAX_KEYWORD_MATCHES]

        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(matched_ids, segments_chosen=list(segment_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_SEGMENTS, df_mean, "VST-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(matched_ids)

submit_button.on_click(on_submit_clicked)
export_button.on_click(on_export_clicked)
keyword_search_button.on_click(on_keyword_search_clicked)

# Layout Setup
print("📊 Gut Expression Dual-Control Workspace Ready:")
display(widgets.VBox([
    glossary_dropdown,
    widgets.HTML("<br><b>Step 1: Choose Gut Segments</b>"),
    widgets.HBox([segment_multiselect, widgets.VBox([cmap_dropdown])]),
    widgets.HTML("<br><b>Step 2: Choose Target Transcripts</b>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<i>Option A: Filter List</i>"), search_filter, gene_multiselect, filter_hint]),
        widgets.VBox([widgets.HTML("<i>Option B: Paste Box</i>"), multi_gene_input]),
        widgets.VBox([widgets.HTML("<i>Option C: Search by Keyword</i>"), keyword_search_box, keyword_search_button]),
    ]),
    widgets.HTML("<br><b>Step 3: Run and/or Save</b>"),
    widgets.HBox([save_location_box, browse_button]),
    widgets.HBox([submit_button, export_button]),
    output_area
]))